In [3]:
!pip install catboost lightgbm -q


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix
)

import joblib
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

In [2]:
os.makedirs("../models/artifacts", exist_ok=True)
os.makedirs("../models/trained_models", exist_ok=True)

In [3]:
df = pd.read_csv("../data/Data.csv", na_values=["", "NA", "NaN"], keep_default_na=True)
df = df.fillna(0)   # ensures no NaNs remain

C:\Users\Raj Bharmani\AppData\Local\Temp\ipykernel_15608\2382404254.py:1: DtypeWarning: Columns (9,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/Data.csv", na_values=["", "NA", "NaN"], keep_default_na=True)


In [4]:
categorical_cols = [col for col in df.columns if df[col].dtype == 'object']
print("Number of Object Columns: ",len(categorical_cols))
for col in categorical_cols:
    df[col] = df[col].astype(str)

Number of Object Columns:  21


In [5]:
def unique_value_distribution(df):
    print("Unique value count per column:\n")
    for col in df.columns:
        unique_vals = df[col].nunique()
        print(f"{col}: {unique_vals} unique values and Dtype: {df[col].dtype}")

In [6]:
unique_value_distribution(df)

Unique value count per column:

BeneID: 138556 unique values and Dtype: object
ClaimID: 558211 unique values and Dtype: object
Provider: 5410 unique values and Dtype: object
InscClaimAmtReimbursed: 438 unique values and Dtype: int64
AttendingPhysician: 82064 unique values and Dtype: object
OperatingPhysician: 35316 unique values and Dtype: object
OtherPhysician: 46458 unique values and Dtype: object
ClmAdmitDiagnosisCode: 4099 unique values and Dtype: object
DeductibleAmtPaid: 17 unique values and Dtype: float64
DiagnosisGroupCode: 1408 unique values and Dtype: object
ClmDiagnosisCode_1: 10451 unique values and Dtype: object
ClmDiagnosisCode_2: 5301 unique values and Dtype: object
ClmDiagnosisCode_3: 4757 unique values and Dtype: object
ClmDiagnosisCode_4: 4360 unique values and Dtype: object
ClmDiagnosisCode_5: 3971 unique values and Dtype: object
ClmDiagnosisCode_6: 3608 unique values and Dtype: object
ClmDiagnosisCode_7: 3389 unique values and Dtype: object
ClmDiagnosisCode_8: 3071 

In [7]:
df = df.drop(columns=["BeneID", "ClaimID", "Provider","AdmissionPeriod"])

In [8]:
df.shape

(558211, 49)

In [9]:
X = df.drop(columns=["PotentialFraud"])
y = df["PotentialFraud"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [23]:
model_columns = X.columns.tolist()
joblib.dump(model_columns, '../models/artifacts/model_columns.pkl')

['../models/artifacts/model_columns.pkl']

In [10]:
print("Data preprocessing completed!")
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Data preprocessing completed!
Training set shape: (446568, 48)
Test set shape: (111643, 48)


In [11]:
# 1. We still need to encode the target variable (y) separately.
target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)
# IMPORTANT: Save this encoder.
joblib.dump(target_encoder, '../models/artifacts/target_encoder.pkl')
print("✅ Target encoder saved!")


# --- THIS IS THE LINE YOU WERE MISSING ---
# 2. Automatically identify all columns that are of 'object' type.
categorical_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']
print(f"Found {len(categorical_cols)} categorical columns to be encoded.")
# --- END OF MISSING LINE ---


# 3. Create the simplified preprocessor using ColumnTransformer.
# This applies OrdinalEncoder to our object columns and leaves the rest alone.
preprocessor = ColumnTransformer(
    transformers=[
        ('cat_encoder', 
         OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), 
         categorical_cols) # This list is now defined from the line above
    ],
    remainder='passthrough'  # This is key: it keeps all other (numerical) columns
)

print(f"✅ Simplified preprocessing pipeline created.")

✅ Target encoder saved!
Found 17 categorical columns to be encoded.
✅ Simplified preprocessing pipeline created.


In [19]:
joblib.dump(categorical_cols, '../models/artifacts/categorical_cols.pkl')
print("✅ Categorical columns list saved!")

✅ Categorical columns list saved!


In [12]:
models = {
    'XGBoost': XGBClassifier(
        n_estimators=2000,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        scale_pos_weight=1.62
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=2000,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
        class_weight={0: 0.81, 1: 1.31}
    ),
    'CatBoost': CatBoostClassifier(
        iterations=2000,
        depth=8,
        learning_rate=0.03,
        subsample=0.8,
        random_seed=42,
        verbose=False,
        thread_count=-1,
        class_weights=[0.81, 1.31]
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    'LogisticRegression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ),
    'SVM': LinearSVC(
        max_iter=2000,
        random_state=42
    )
}

In [13]:
results = {}
trained_pipelines = {}

In [14]:
for model_name, model in tqdm(models.items(), desc="Training model pipelines"):
    # Create the full pipeline: Preprocess -> Scale -> Model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    print(f"\nTraining {model_name} pipeline...")

    # Fit the entire pipeline on the RAW training data
    pipeline.fit(X_train, y_train_encoded)

    # Make predictions using the test set
    y_pred = pipeline.predict(X_test)
    if hasattr(pipeline, "predict_proba"):
        y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    else: # For models like SVM
        y_pred_proba = pipeline.decision_function(X_test)

    # Store results and the fitted pipeline
    results[model_name] = {
        'accuracy': accuracy_score(y_test_encoded, y_pred),
        'precision': precision_recall_fscore_support(y_test_encoded, y_pred, average='binary')[0],
        'recall': precision_recall_fscore_support(y_test_encoded, y_pred, average='binary')[1],
        'f1': precision_recall_fscore_support(y_test_encoded, y_pred, average='binary')[2],
        'auc_roc': roc_auc_score(y_test_encoded, y_pred_proba)
    }
    trained_pipelines[model_name] = pipeline
    print(f"✅ {model_name} pipeline training complete!")

Training model pipelines:   0%|                                                                  | 0/6 [00:00<?, ?it/s]


Training XGBoost pipeline...


Training model pipelines:  17%|█████████▌                                               | 1/6 [04:54<24:31, 294.34s/it]

✅ XGBoost pipeline training complete!

Training LightGBM pipeline...


C:\Users\Raj Bharmani\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Raj Bharmani\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Training model pipelines:  33%|███████████████████                                      | 2/6 [07:35<14:22, 215.74s/it]

...

✅ RandomForest pipeline training complete!

Training LogisticRegression pipeline...


Training model pipelines:  83%|███████████████████████████████████████████████▌         | 5/6 [21:37<03:33, 213.55s/it]

✅ LogisticRegression pipeline training complete!

Training SVM pipeline...


Training model pipelines: 100%|█████████████████████████████████████████████████████████| 6/6 [22:03<00:00, 220.62s/it]

✅ SVM pipeline training complete!


In [15]:
print("\n" + "="*80)
print("MODEL COMPARISON RESULTS")
print("="*80)

comparison_df = pd.DataFrame()
for model_name, result in results.items():
    comparison_df = pd.concat([comparison_df, pd.DataFrame({
        'Model': [model_name],
        'Accuracy': [f"{result['accuracy']:.4f}"],
        'Precision': [f"{result['precision']:.4f}"],
        'Recall': [f"{result['recall']:.4f}"],
        'F1-Score': [f"{result['f1']:.4f}"],
        'AUC-ROC': [f"{result['auc_roc']:.4f}"]
    })], ignore_index=True)

print(comparison_df.to_string(index=False))


MODEL COMPARISON RESULTS
             Model Accuracy Precision Recall F1-Score AUC-ROC
           XGBoost   0.8414    0.7936 0.7893   0.7914  0.9108
          LightGBM   0.8134    0.7511 0.7638   0.7574  0.8849
          CatBoost   0.7999    0.7295 0.7551   0.7420  0.8702
      RandomForest   0.6596    0.6765 0.2054   0.3151  0.6788
LogisticRegression   0.6308    0.5823 0.1111   0.1867  0.5502
               SVM   0.6308    0.5822 0.1112   0.1867  0.5501


In [16]:
print("SAVING ALL TRAINED PIPELINES...")
for model_name, pipeline in tqdm(trained_pipelines.items(), desc="Saving pipelines"):
    # The filename should now indicate it's a full pipeline
    pipeline_filename = f'../models/trained_models/{model_name.lower()}_pipeline.pkl'
    joblib.dump(pipeline, pipeline_filename)
    print(f"✅ {model_name} pipeline saved as {pipeline_filename}")

print("\nAll pipelines saved successfully!")

SAVING ALL TRAINED PIPELINES...


Saving pipelines:  17%|███████████                                                       | 1/6 [00:00<00:02,  1.84it/s]

✅ XGBoost pipeline saved as ../models/trained_models/xgboost_pipeline.pkl


Saving pipelines:  50%|█████████████████████████████████                                 | 3/6 [00:01<00:01,  2.82it/s]

✅ LightGBM pipeline saved as ../models/trained_models/lightgbm_pipeline.pkl
✅ CatBoost pipeline saved as ../models/trained_models/catboost_pipeline.pkl


Saving pipelines: 100%|██████████████████████████████████████████████████████████████████| 6/6 [00:01<00:00,  3.31it/s]

✅ RandomForest pipeline saved as ../models/trained_models/randomforest_pipeline.pkl
✅ LogisticRegression pipeline saved as ../models/trained_models/logisticregression_pipeline.pkl
✅ SVM pipeline saved as ../models/trained_models/svm_pipeline.pkl

All pipelines saved successfully!


Testing Neural Network 

In [28]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

# Set random seeds for reproducibility
tf.random.set_seed(42)

In [29]:
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
print(f"Class weights: {class_weight_dict}")

Class weights: {0: 0.808028024260672, 1: 1.3116144646255785}


In [35]:
def create_neural_network(input_dim):
    model = Sequential([
        # Input layer
        Dense(512, activation='relu', input_dim=input_dim),
        BatchNormalization(),
        Dropout(0.4),

        # Hidden layers
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(32, activation='relu'),
        Dropout(0.4),

        # Output layer
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )

    return model

# Create the model
input_dim = X_train_scaled.shape[1]
nn_model = create_neural_network(input_dim)

print("\nNeural Network Architecture:")
nn_model.summary()


Neural Network Architecture:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 512)            │        25,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,521 (795.00 KB)

 Trainable params: 201,601 (787.50 KB)

 Non-trainable params: 1,920 (7.50 KB)

In [36]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    )
]


In [37]:
class TrainingProgressCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.pbar = None

    def on_train_begin(self, logs=None):
        self.pbar = tqdm(total=self.params['epochs'], desc="Training Neural Network")

    def on_epoch_end(self, epoch, logs=None):
        self.pbar.update(1)
        self.pbar.set_postfix({
            'loss': f"{logs.get('loss', 0):.4f}",
            'val_loss': f"{logs.get('val_loss', 0):.4f}",
            'val_accuracy': f"{logs.get('val_accuracy', 0):.4f}"
        })

    def on_train_end(self, logs=None):
        self.pbar.close()

progress_callback = TrainingProgressCallback()
callbacks.append(progress_callback)



In [ ]:
print(f"\nStarting Neural Network training...")
print(f"Training data shape: {X_train_scaled.shape}")
print(f"Validation data shape: {X_test_scaled.shape}")
history = nn_model.fit(
    X_train_scaled, y_train_encoded,
    validation_data=(X_test_scaled, y_test_encoded),
    epochs=100,
    batch_size=1024,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=0
)

print("\nNeural Network training completed!")

In [ ]:
nn_predictions_proba = nn_model.predict(X_test_scaled, verbose=0).flatten()
nn_predictions = (nn_predictions_proba > 0.5).astype(int)

# Calculate metrics
nn_accuracy = accuracy_score(y_test_encoded, nn_predictions)
nn_precision, nn_recall, nn_f1, _ = precision_recall_fscore_support(y_test_encoded, nn_predictions, average='binary')
nn_auc_roc = roc_auc_score(y_test_encoded, nn_predictions_proba)

# Add Neural Network results to comparison
nn_result = {
    'model': nn_model,
    'accuracy': nn_accuracy,
    'precision': nn_precision,
    'recall': nn_recall,
    'f1': nn_f1,
    'auc_roc': nn_auc_roc,
    'predictions': nn_predictions,
    'probabilities': nn_predictions_proba
}

results['NeuralNetwork'] = nn_result

print(f"\nNeural Network Results:")
print(f"Accuracy: {nn_accuracy:.4f}")
print(f"Precision: {nn_precision:.4f}")
print(f"Recall: {nn_recall:.4f}")
print(f"F1-Score: {nn_f1:.4f}")
print(f"AUC-ROC: {nn_auc_roc:.4f}")


Neural Network Results:
Accuracy: 0.7975
Precision: 0.7892
Recall: 0.6396
F1-Score: 0.7066
AUC-ROC: 0.8463
